In [3]:
import pandas as pd
import numpy as np
import os
from pytrends.request import TrendReq
import time
#pd.set_option('future.no_silent_downcasting', True)
path = os.getcwd()
path_source = f"{path}/../source"


dic = pd.read_excel(f"{path_source}/Google Search.xlsx")
dic['location'] = dic['location'].fillna('')

In [5]:
def gt( kw , g , t='today 5-y', retry=5 ) :
    r = 1
    while r <= retry :
        try :
            pytrends = TrendReq(hl='en-US', tz=360)
            pytrends.build_payload([kw], geo=g, timeframe=t)
            data = pytrends.interest_over_time()
            r = retry+100
            try :
                data = data[kw]
                return data
            except :
                return pd.DataFrame()
            
        except :
            print(f"Tryout {r}. word: {kw}. Sleep time activated...")
            if r <= 2 : time.sleep(30)
            else : time.sleep(120)
            r = r + 1

    time.sleep(int(np.random.rand(1)[0]*10))

target = dic

In [7]:
try :
    len(results) > 0 == True
except :
    results = []
from tqdm import tqdm 

for _, row in tqdm(target.iterrows(), total=target.shape[0], position=0, leave=True):
    print(f"{row['keyword']} {row['location']}")
    try :
        df = pd.concat( [ r for r in results if len(r)>0 ] , axis=1)
        if f"{row['keyword']} {row['location']}" in df.columns : continue
    except :
        True
    
    df = pd.DataFrame(gt(row['keyword'], row['location'], t="all"))
    df.rename(columns = { row['keyword'] : f"{row['keyword']} {row['location']}" } , inplace=True)
    results.append(df)

  0%|          | 0/104 [00:00<?, ?it/s]

bahamas flights US
Tryout 1. word: bahamas flights. Sleep time activated...
Tryout 2. word: bahamas flights. Sleep time activated...


  0%|          | 0/104 [01:01<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import os
from pytrends.request import TrendReq
import time
from random import randint

path = os.getcwd()
path_source = f"{path}/../source"
dic = pd.read_excel(f"{path_source}/Google Search.xlsx")
dic['location'] = dic['location'].fillna('')

def gt(kw, g, t='today 5-y', retry=5):
    r = 1
    while r <= retry:
        try:
            # Add timeout and increase backoff_factor
            pytrends = TrendReq(hl='en-US', tz=360, timeout=(10, 25), retries=2, backoff_factor=0.5)
            pytrends.build_payload([kw], geo=g, timeframe=t)
            data = pytrends.interest_over_time()
            
            # Add delay even on success to avoid rate limits
            time.sleep(randint(2, 5))
            
            if data.empty or kw not in data.columns:
                return pd.DataFrame()
            
            return data[kw]
            
        except Exception as e:
            print(f"Tryout {r}/{retry}. word: {kw}. Error: {str(e)}")
            if r < retry:
                # Exponential backoff
                wait_time = min(300, 30 * (2 ** (r - 1)))  # Cap at 5 minutes
                print(f"Waiting {wait_time} seconds...")
                time.sleep(wait_time)
            r += 1
    
    print(f"Failed after {retry} attempts for: {kw}")
    return pd.DataFrame()

target = dic

# Initialize or load existing results
results = []
try:
    # Try to load previous results if script was interrupted
    if os.path.exists(f"{path}/google_trends_results.pkl"):
        results = pd.read_pickle(f"{path}/google_trends_results.pkl")
        print(f"Loaded {len(results)} previous results")
except:
    results = []

from tqdm import tqdm 

for idx, row in tqdm(target.iterrows(), total=target.shape[0], position=0, leave=True):
    kw_location = f"{row['keyword']} {row['location']}"
    print(f"\n{kw_location}")
    
    # Check if already processed
    try:
        if results:
            df_check = pd.concat([r for r in results if not r.empty], axis=1)
            if kw_location in df_check.columns:
                print(f"Skipping (already processed)")
                continue
    except:
        pass
    
    df = gt(row['keyword'], row['location'], t="all")
    
    if not df.empty:
        df = pd.DataFrame(df)
        df.rename(columns={row['keyword']: kw_location}, inplace=True)
        results.append(df)
        
        # Save progress periodically
        if len(results) % 10 == 0:
            pd.to_pickle(results, f"{path}/google_trends_results_backup.pkl")
            print(f"Progress saved ({len(results)} queries)")
    else:
        print(f"No data returned for {kw_location}")
    
    # Random delay between queries
    time.sleep(randint(3, 8))

# Save final results
try:
    final_df = pd.concat([r for r in results if not r.empty], axis=1)
    final_df.to_csv(f"{path}/google_trends_results.csv")
    pd.to_pickle(results, f"{path}/google_trends_results.pkl")
    print(f"\nCompleted! Saved {len(results)} queries")
except Exception as e:
    print(f"Error saving: {e}")

  0%|          | 0/104 [00:00<?, ?it/s]


bahamas flights US
Tryout 1/5. word: bahamas flights. Error: Retry.__init__() got an unexpected keyword argument 'method_whitelist'
Waiting 30 seconds...


In [9]:
pytrends = TrendReq(hl='en-US', tz=360)
pytrends.build_payload([ row['keyword'] ], geo=row['location'], timeframe="all")
data = pytrends.interest_over_time()

TooManyRequestsError: The request failed: Google returned a response with code 429

In [58]:
df = pd.concat( [ r for r in results if len(r)>0 ] , axis=1)
df

,bahamas hotels US,bahamas cruise US,bahamas flights,bahamas flights US,bahamas hotels,bahamas cruise,restaurant BS,jobs BS,kraven BS,Linkedin BS,242 Jobs BS,Newspapers BS,Ministry Labour BS
date,,,,,,,,,,,,,
2004-01-01,100,49,55,26,100,75,0,0,0,0,0,0,0
2004-02-01,84,45,52,30,84,61,0,0,0,0,0,0,0
2004-03-01,79,42,34,21,70,62,0,0,0,0,0,0,0
2004-04-01,77,38,42,17,81,56,0,0,0,0,0,0,0
2004-05-01,66,42,43,23,66,58,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-01-01,40,79,80,78,27,70,87,56,74,23,21,0,0
2025-02-01,39,66,68,76,25,64,100,52,50,22,29,0,0
2025-03-01,35,69,65,67,23,64,88,52,37,25,25,0,0


In [ ]:
df.to_csv(f'{path_source}/gtrends.csv')

In [ ]:
df['Cost Right BS'].plot()